# Phase 4 — Baseline Models
## UNSW-NB15 GNN-based NIDS Pipeline

Phases 1–3 ✅ complete (EDA, Preprocessing, Graph Construction).  
This notebook trains and evaluates **5 baseline classifiers** with hyperparameter tuning on the UNSW-NB15 network intrusion detection task.

| Item | Detail |
|---|---|
| **Models** | Logistic Regression, Random Forest, XGBoost, LightGBM, MLP (PyTorch) |
| **Tasks** | Binary (Normal vs Attack) + Multiclass (10 attack categories) |
| **HPO** | `RandomizedSearchCV` on 200 k stratified subsample (sklearn/XGB/LGBM); manual arch search (MLP) |
| **Train data** | Binary: `X_train_binary_resampled` (ROS 2:1, 2.35 M rows); Multiclass: `X_train_multiclass_resampled` (SMOTE, 1.66 M rows) |
| **Eval** | Unaugmented `X_val` (411 k) + `X_test` (82 k) only |
| **Features** | 37 RobustScaled float32 features |

> ⚠️ **Distribution shift:** Val attack ratio = 4.84% | Test attack ratio = 55.06% — report both and note the shift.

### Multiclass label map
```
{0: Analysis, 1: Backdoor, 2: DoS, 3: Exploits, 4: Fuzzers,
 5: Generic, 6: Normal, 7: Reconnaissance, 8: Shellcode, 9: Worms}
```

In [ ]:
import os

# ── One variable to switch platforms ──────────────────────────────────────────────
PLATFORM = 'colab'  # 'local' | 'kaggle' | 'colab'
# ────────────────────────────────────────────────────────────────────────────────

if PLATFORM == 'kaggle':
    BASE    = '/kaggle/input/unsw-nb15-preprocessed'
    OUT_DIR = '/kaggle/working/outputs/models'
    OUT_PRE = f'{BASE}/preprocessed'
    OUT_ENC = f'{BASE}/encoders'

elif PLATFORM == 'colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print('WARNING: google.colab not available — continuing without Drive mount.')
    BASE    = '/content/drive/MyDrive/outputs'
    OUT_DIR = f'{BASE}/models'
    OUT_PRE = f'{BASE}/preprocessed'
    OUT_ENC = f'{BASE}/encoders'

else:  # local
    BASE    = r'c:\Users\Asus\OneDrive\Desktop\GNN\UNSW-NB15'
    OUT_DIR = os.path.join(BASE, 'outputs', 'models')
    OUT_PRE = os.path.join(BASE, 'outputs', 'preprocessed')
    OUT_ENC = os.path.join(BASE, 'outputs', 'encoders')

os.makedirs(OUT_DIR, exist_ok=True)
print(f'Platform     : {PLATFORM}')
print(f'Preprocessed : {OUT_PRE}')
print(f'Encoders     : {OUT_ENC}')
print(f'Models out   : {OUT_DIR}')

In [ ]:
# Install / verify packages (Colab / Kaggle)
import subprocess, sys

if PLATFORM == 'colab':
    pkgs = ['xgboost', 'lightgbm', 'imbalanced-learn']
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'], check=True)
    print('Packages installed/verified.')
elif PLATFORM == 'kaggle':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)
    print('torch_geometric installed.')
else:
    print('Local environment — assuming packages pre-installed.')

In [ ]:
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_GPU = torch.cuda.is_available()

print(f'NumPy   : {np.__version__}')
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {USE_GPU}  →  device={DEVICE}')
if USE_GPU:
    print(f'GPU name: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Load all preprocessed arrays ───────────────────────────────────────────────────
print('Loading data...')
t0 = time.time()

X_train     = np.load(os.path.join(OUT_PRE, 'X_train.npy'))
X_val       = np.load(os.path.join(OUT_PRE, 'X_val.npy'))
X_test      = np.load(os.path.join(OUT_PRE, 'X_test.npy'))

X_bin_res   = np.load(os.path.join(OUT_PRE, 'X_train_binary_resampled.npy'))
X_mc_res    = np.load(os.path.join(OUT_PRE, 'X_train_multiclass_resampled.npy'))

y_bin_train = np.load(os.path.join(OUT_PRE, 'y_binary_train.npy'))
y_bin_res   = np.load(os.path.join(OUT_PRE, 'y_binary_train_resampled.npy'))
y_bin_val   = np.load(os.path.join(OUT_PRE, 'y_binary_val.npy'))
y_bin_test  = np.load(os.path.join(OUT_PRE, 'y_binary_test.npy'))

y_mc_train  = np.load(os.path.join(OUT_PRE, 'y_multiclass_train.npy'))
y_mc_res    = np.load(os.path.join(OUT_PRE, 'y_multiclass_train_resampled.npy'))
y_mc_val    = np.load(os.path.join(OUT_PRE, 'y_multiclass_val.npy'))
y_mc_test   = np.load(os.path.join(OUT_PRE, 'y_multiclass_test.npy'))

with open(os.path.join(OUT_PRE, 'feature_names.json')) as f:
    feature_names = json.load(f)

le_attack_cat = joblib.load(os.path.join(OUT_ENC, 'le_attack_cat.pkl'))
class_names   = list(le_attack_cat.classes_)  # index 0–9 → label names

print(f'Loaded in {time.time()-t0:.1f}s')
print(f'X_train       : {X_train.shape}   dtype={X_train.dtype}')
print(f'X_bin_res     : {X_bin_res.shape}  (binary resampled 2:1)')
print(f'X_mc_res      : {X_mc_res.shape}   (multiclass SMOTE)')
print(f'X_val         : {X_val.shape}')
print(f'X_test        : {X_test.shape}')
print(f'Features ({len(feature_names)}): {feature_names[:5]} ...')
print(f'Class names   : {class_names}')

In [ ]:
# ── Shared helpers ─────────────────────────────────────────────────────────────────

def evaluate_binary(model, X, y, split_name='val', model_name='model'):
    """Compute binary classification metrics and return as dict."""
    y_pred = model.predict(X)
    fpr_val = float((y_pred[y == 0] == 1).sum() / max((y == 0).sum(), 1))
    m = {
        'accuracy' : float(accuracy_score(y, y_pred)),
        'precision': float(precision_score(y, y_pred, zero_division=0)),
        'recall'   : float(recall_score(y, y_pred, zero_division=0)),
        'f1'       : float(f1_score(y, y_pred, zero_division=0)),
        'fpr'      : fpr_val,
    }
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X)[:, 1]
        m['roc_auc'] = float(roc_auc_score(y, y_prob))
    elif hasattr(model, 'decision_function'):
        m['roc_auc'] = float(roc_auc_score(y, model.decision_function(X)))
    else:
        m['roc_auc'] = float('nan')
    print(f'  [{model_name:<16} | {split_name}]  '
          f'Acc={m["accuracy"]:.4f}  P={m["precision"]:.4f}  R={m["recall"]:.4f}  '
          f'F1={m["f1"]:.4f}  AUC={m["roc_auc"]:.4f}  FPR={m["fpr"]:.4f}')
    return m


def evaluate_multiclass(model, X, y, split_name='val', model_name='model'):
    """Compute multiclass metrics and return as dict."""
    y_pred = model.predict(X)
    per_f1 = f1_score(y, y_pred, average=None, zero_division=0)
    m = {
        'f1_macro'    : float(f1_score(y, y_pred, average='macro',    zero_division=0)),
        'f1_weighted' : float(f1_score(y, y_pred, average='weighted', zero_division=0)),
        'accuracy'    : float(accuracy_score(y, y_pred)),
        'per_class_f1': {class_names[i]: float(per_f1[i])
                         for i in range(min(len(per_f1), len(class_names)))},
    }
    print(f'  [{model_name:<16} | {split_name}]  '
          f'Acc={m["accuracy"]:.4f}  F1-macro={m["f1_macro"]:.4f}  '
          f'F1-weighted={m["f1_weighted"]:.4f}')
    return m


def stratified_subsample(X, y, n=200_000, random_state=42):
    """Stratified subsample of n rows from (X, y). Returns full data if n >= len(y)."""
    if len(y) <= n:
        return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=random_state)
    idx, _ = next(sss.split(X, y))
    print(f'    Subsampled {len(y):,} → {len(idx):,} rows (stratified, n={n:,})')
    return X[idx], y[idx]


def _serialisable(obj):
    """Recursively convert numpy/torch types to JSON-serialisable Python types."""
    if isinstance(obj, dict):
        return {k: _serialisable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_serialisable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# Result stores
results  = {}   # {model_name: {'val': {...}, 'test': {...}, 'best_params': {...}}}
models   = {}   # {model_name: fitted model or wrapper}
roc_data = {}   # {label: (fpr_arr, tpr_arr, auc_float)} for binary models

SUBSAMPLE_N = 200_000
print('Helper functions ready. SUBSAMPLE_N =', SUBSAMPLE_N)

---
## § 4.1 — Logistic Regression

HPO via `RandomizedSearchCV(n_iter=8, cv=3)` on a **200 k stratified subsample** of the resampled data, then refit on the full resampled set with best parameters.

- Binary: `LogisticRegression(class_weight='balanced', solver=best, C=best)`
- Multiclass: same with `multi_class='multinomial'`

> LR fits quickly, but on 2.35 M rows with `saga` it can still take a few minutes. HPO on 200 k keeps wall-clock under 5 min.

In [ ]:
# ── 4.1a  LR Binary ──────────────────────────────────────────────────────────────────────
t0 = time.time()
print('\n► LR Binary — HPO + Training')

X_lr, y_lr = stratified_subsample(X_bin_res, y_bin_res, n=SUBSAMPLE_N)

param_dist_lr_bin = {
    'C'      : [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
    'solver' : ['saga', 'lbfgs'],
    'penalty': ['l2'],
}
search_lr_bin = RandomizedSearchCV(
    LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=-1, random_state=SEED),
    param_dist_lr_bin, n_iter=8, cv=3,
    scoring='f1', n_jobs=-1, random_state=SEED, verbose=1
)
search_lr_bin.fit(X_lr, y_lr)
best_lr_bin = search_lr_bin.best_params_
print(f'  Best params (LR binary): {best_lr_bin}')

lr_binary = LogisticRegression(
    **best_lr_bin, max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=SEED
)
print(f'  Refitting on full resampled data ({len(y_bin_res):,} rows)...')
lr_binary.fit(X_bin_res, y_bin_res)

results['lr_binary'] = {
    'val'        : evaluate_binary(lr_binary, X_val,  y_bin_val,  'val',  'LR-Binary'),
    'test'       : evaluate_binary(lr_binary, X_test, y_bin_test, 'test', 'LR-Binary'),
    'best_params': best_lr_bin,
}
models['lr_binary'] = lr_binary

y_prob_lr = lr_binary.predict_proba(X_val)[:, 1]
fpr_lr, tpr_lr, _ = roc_curve(y_bin_val, y_prob_lr)
roc_data['LR'] = (fpr_lr, tpr_lr, results['lr_binary']['val']['roc_auc'])
print(f'\u2713 LR Binary done in {time.time()-t0:.1f}s')

In [ ]:
# ── 4.1b  LR Multiclass ─────────────────────────────────────────────────────────────────
t0 = time.time()
print('\n► LR Multiclass — HPO + Training')

X_lr_mc, y_lr_mc = stratified_subsample(X_mc_res, y_mc_res, n=SUBSAMPLE_N)

param_dist_lr_mc = {
    'C'      : [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
    'solver' : ['saga', 'lbfgs'],
}
search_lr_mc = RandomizedSearchCV(
    LogisticRegression(max_iter=500, class_weight='balanced',
                       multi_class='multinomial', n_jobs=-1, random_state=SEED),
    param_dist_lr_mc, n_iter=8, cv=3,
    scoring='f1_macro', n_jobs=-1, random_state=SEED, verbose=1
)
search_lr_mc.fit(X_lr_mc, y_lr_mc)
best_lr_mc = search_lr_mc.best_params_
print(f'  Best params (LR multiclass): {best_lr_mc}')

lr_multiclass = LogisticRegression(
    **best_lr_mc, max_iter=1000, class_weight='balanced',
    multi_class='multinomial', n_jobs=-1, random_state=SEED
)
print(f'  Refitting on full resampled data ({len(y_mc_res):,} rows)...')
lr_multiclass.fit(X_mc_res, y_mc_res)

results['lr_multiclass'] = {
    'val'        : evaluate_multiclass(lr_multiclass, X_val,  y_mc_val,  'val',  'LR-Multiclass'),
    'test'       : evaluate_multiclass(lr_multiclass, X_test, y_mc_test, 'test', 'LR-Multiclass'),
    'best_params': best_lr_mc,
}
models['lr_multiclass'] = lr_multiclass
print(f'\u2713 LR Multiclass done in {time.time()-t0:.1f}s')

---
## § 4.2 — Random Forest

HPO via `RandomizedSearchCV(n_iter=8, cv=3)` on 200 k subsample — RF is slow on millions of rows.  
Refit on full resampled data with best parameters.

> **Flagged:** HPO and full training both use subsampling / `class_weight='balanced'` to manage wall-clock time.

In [ ]:
# ── 4.2a  RF Binary ───────────────────────────────────────────────────────────────────────
t0 = time.time()
print('\n► RF Binary — HPO (200k subsample) + Full Training')

X_rf, y_rf = stratified_subsample(X_bin_res, y_bin_res, n=SUBSAMPLE_N)

param_dist_rf = {
    'n_estimators'     : [100, 200, 300, 400],
    'max_depth'        : [None, 15, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'max_features'     : ['sqrt', 'log2', 0.25, 0.33],
    'min_samples_leaf' : [1, 2, 4],
}
search_rf_bin = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=SEED),
    param_dist_rf, n_iter=8, cv=3,
    scoring='f1', n_jobs=1, random_state=SEED, verbose=1
)
search_rf_bin.fit(X_rf, y_rf)
best_rf_bin = search_rf_bin.best_params_
print(f'  Best params (RF binary): {best_rf_bin}')

rf_binary = RandomForestClassifier(
    **best_rf_bin, class_weight='balanced', n_jobs=-1, random_state=SEED
)
print(f'  Refitting on full resampled data ({len(y_bin_res):,} rows)...')
rf_binary.fit(X_bin_res, y_bin_res)

results['rf_binary'] = {
    'val'        : evaluate_binary(rf_binary, X_val,  y_bin_val,  'val',  'RF-Binary'),
    'test'       : evaluate_binary(rf_binary, X_test, y_bin_test, 'test', 'RF-Binary'),
    'best_params': best_rf_bin,
}
models['rf_binary'] = rf_binary

# Guard: feature_names.json may contain fewer names than actual feature count.
# Fall back to generic labels if lengths don't match.
n_features = X_bin_res.shape[1]
feat_labels = (feature_names
               if len(feature_names) == n_features
               else [f'feat_{i}' for i in range(n_features)])
if len(feature_names) != n_features:
    print(f'  ⚠ feature_names has {len(feature_names)} entries but model has {n_features} '
          f'features — using generic labels feat_0…feat_{n_features-1}')

rf_importances = pd.Series(
    rf_binary.feature_importances_, index=feat_labels
).sort_values(ascending=False)

y_prob_rf = rf_binary.predict_proba(X_val)[:, 1]
fpr_rf, tpr_rf, _ = roc_curve(y_bin_val, y_prob_rf)
roc_data['RF'] = (fpr_rf, tpr_rf, results['rf_binary']['val']['roc_auc'])
print(f'  Top-5 features: {rf_importances.head().to_dict()}')
print(f'✓ RF Binary done in {time.time()-t0:.1f}s')

In [ ]:
# ── 4.2b  RF Multiclass ─────────────────────────────────────────────────────────────────
t0 = time.time()
print('\n► RF Multiclass — HPO (200k subsample) + Full Training')

X_rf_mc, y_rf_mc = stratified_subsample(X_mc_res, y_mc_res, n=SUBSAMPLE_N)

search_rf_mc = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=SEED),
    param_dist_rf, n_iter=8, cv=3,
    scoring='f1_macro', n_jobs=1, random_state=SEED, verbose=1
)
search_rf_mc.fit(X_rf_mc, y_rf_mc)
best_rf_mc = search_rf_mc.best_params_
print(f'  Best params (RF multiclass): {best_rf_mc}')

rf_multiclass = RandomForestClassifier(
    **best_rf_mc, class_weight='balanced', n_jobs=-1, random_state=SEED
)
print(f'  Refitting on full resampled data ({len(y_mc_res):,} rows)...')
rf_multiclass.fit(X_mc_res, y_mc_res)

results['rf_multiclass'] = {
    'val'        : evaluate_multiclass(rf_multiclass, X_val,  y_mc_val,  'val',  'RF-Multiclass'),
    'test'       : evaluate_multiclass(rf_multiclass, X_test, y_mc_test, 'test', 'RF-Multiclass'),
    'best_params': best_rf_mc,
}
models['rf_multiclass'] = rf_multiclass
print(f'\u2713 RF Multiclass done in {time.time()-t0:.1f}s')

---
## § 4.3 — XGBoost

HPO via `RandomizedSearchCV(n_iter=10, cv=3)` on 200 k subsample.  
Final training on full resampled data with `early_stopping_rounds=50` monitored on val set.

- **Device:** `cuda` if GPU available, else `cpu`
- **Binary:** `objective='binary:logistic'`, `scale_pos_weight=2`
- **Multiclass:** `objective='multi:softmax'`, `num_class=10` (labels must be 0-indexed integers ✔)

In [ ]:
# ── 4.3a  XGBoost Binary ──────────────────────────────────────────────────────────────────
t0 = time.time()
DEVICE_XGB = 'cuda' if USE_GPU else 'cpu'
print(f'\n► XGBoost Binary — HPO + Training  (device={DEVICE_XGB})')

X_xgb, y_xgb = stratified_subsample(X_bin_res, y_bin_res, n=SUBSAMPLE_N)

param_dist_xgb_bin = {
    'max_depth'        : [4, 5, 6, 7, 8, 10],
    'learning_rate'    : [0.03, 0.05, 0.1, 0.15, 0.2],
    'n_estimators'     : [200, 300, 400, 500],
    'subsample'        : [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree' : [0.6, 0.7, 0.8, 1.0],
    'min_child_weight' : [1, 3, 5, 7],
    'gamma'            : [0, 0.1, 0.2, 0.5],
    'reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'reg_lambda'       : [0.5, 1.0, 2.0],
}
search_xgb_bin = RandomizedSearchCV(
    XGBClassifier(
        objective='binary:logistic', scale_pos_weight=2,
        tree_method='hist', device=DEVICE_XGB,
        eval_metric='logloss', random_state=SEED, verbosity=0
    ),
    param_dist_xgb_bin, n_iter=10, cv=3,
    scoring='f1', n_jobs=1, random_state=SEED, verbose=1
)
search_xgb_bin.fit(X_xgb, y_xgb)
best_xgb_bin = search_xgb_bin.best_params_
print(f'  Best params (XGB binary): {best_xgb_bin}')

xgb_binary = XGBClassifier(
    **best_xgb_bin,
    objective='binary:logistic', scale_pos_weight=2,
    tree_method='hist', device=DEVICE_XGB,
    early_stopping_rounds=50,
    eval_metric='logloss', random_state=SEED, verbosity=0
)
print(f'  Refitting on full resampled data ({len(y_bin_res):,} rows) with early stopping...')
xgb_binary.fit(
    X_bin_res, y_bin_res,
    eval_set=[(X_val, y_bin_val)],
    verbose=100
)

results['xgb_binary'] = {
    'val'        : evaluate_binary(xgb_binary, X_val,  y_bin_val,  'val',  'XGB-Binary'),
    'test'       : evaluate_binary(xgb_binary, X_test, y_bin_test, 'test', 'XGB-Binary'),
    'best_params': best_xgb_bin,
}
models['xgb_binary'] = xgb_binary

y_prob_xgb = xgb_binary.predict_proba(X_val)[:, 1]
fpr_xgb, tpr_xgb, _ = roc_curve(y_bin_val, y_prob_xgb)
roc_data['XGBoost'] = (fpr_xgb, tpr_xgb, results['xgb_binary']['val']['roc_auc'])
print(f'\u2713 XGBoost Binary done in {time.time()-t0:.1f}s')

In [ ]:
# ── 4.3b  XGBoost Multiclass ─────────────────────────────────────────────────────────────
t0 = time.time()
print(f'\n► XGBoost Multiclass — HPO + Training  (device={DEVICE_XGB})')

X_xgb_mc, y_xgb_mc = stratified_subsample(X_mc_res, y_mc_res, n=SUBSAMPLE_N)
y_xgb_mc_int = y_xgb_mc.astype(int)

param_dist_xgb_mc = {
    'max_depth'        : [4, 5, 6, 7, 8],
    'learning_rate'    : [0.03, 0.05, 0.1, 0.15, 0.2],
    'n_estimators'     : [200, 300, 400, 500],
    'subsample'        : [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree' : [0.6, 0.8, 1.0],
    'min_child_weight' : [1, 3, 5],
    'reg_alpha'        : [0, 0.01, 0.1],
    'reg_lambda'       : [0.5, 1.0, 2.0],
}
search_xgb_mc = RandomizedSearchCV(
    XGBClassifier(
        objective='multi:softmax', num_class=10,
        tree_method='hist', device=DEVICE_XGB,
        eval_metric='mlogloss', random_state=SEED, verbosity=0
    ),
    param_dist_xgb_mc, n_iter=10, cv=3,
    scoring='f1_macro', n_jobs=1, random_state=SEED, verbose=1
)
search_xgb_mc.fit(X_xgb_mc, y_xgb_mc_int)
best_xgb_mc = search_xgb_mc.best_params_
print(f'  Best params (XGB multiclass): {best_xgb_mc}')

xgb_multiclass = XGBClassifier(
    **best_xgb_mc,
    objective='multi:softmax', num_class=10,
    tree_method='hist', device=DEVICE_XGB,
    early_stopping_rounds=50,
    eval_metric='mlogloss', random_state=SEED, verbosity=0
)
print(f'  Refitting on full resampled data ({len(y_mc_res):,} rows) with early stopping...')
xgb_multiclass.fit(
    X_mc_res, y_mc_res.astype(int),
    eval_set=[(X_val, y_mc_val.astype(int))],
    verbose=100
)

results['xgb_multiclass'] = {
    'val'        : evaluate_multiclass(xgb_multiclass, X_val,  y_mc_val,  'val',  'XGB-Multiclass'),
    'test'       : evaluate_multiclass(xgb_multiclass, X_test, y_mc_test, 'test', 'XGB-Multiclass'),
    'best_params': best_xgb_mc,
}
models['xgb_multiclass'] = xgb_multiclass
print(f'\u2713 XGBoost Multiclass done in {time.time()-t0:.1f}s')

---
## § 4.4 — LightGBM

HPO via `RandomizedSearchCV(n_iter=10, cv=3)` on 200 k subsample.  
Final training on full resampled data with `lgb.early_stopping(50)` monitored on val set.

In [ ]:
# ── 4.4a  LightGBM Binary ─────────────────────────────────────────────────────────────────
t0 = time.time()
print('\n► LightGBM Binary — HPO + Training')

X_lgb, y_lgb = stratified_subsample(X_bin_res, y_bin_res, n=SUBSAMPLE_N)

param_dist_lgb_bin = {
    'num_leaves'       : [31, 63, 127, 255],
    'learning_rate'    : [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    'n_estimators'     : [200, 300, 400, 500, 700],
    'subsample'        : [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree' : [0.6, 0.7, 0.8, 1.0],
    'min_child_samples': [10, 20, 50, 100],
    'reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'reg_lambda'       : [0, 0.1, 1.0, 5.0],
}
search_lgb_bin = RandomizedSearchCV(
    LGBMClassifier(class_weight='balanced', n_jobs=-1, random_state=SEED, verbose=-1),
    param_dist_lgb_bin, n_iter=10, cv=3,
    scoring='f1', n_jobs=1, random_state=SEED, verbose=1
)
search_lgb_bin.fit(X_lgb, y_lgb)
best_lgb_bin = search_lgb_bin.best_params_
print(f'  Best params (LGBM binary): {best_lgb_bin}')

lgbm_binary = LGBMClassifier(
    **best_lgb_bin, class_weight='balanced', n_jobs=-1, random_state=SEED, verbose=-1
)
print(f'  Refitting on full resampled data ({len(y_bin_res):,} rows) with early stopping...')
lgbm_binary.fit(
    X_bin_res, y_bin_res,
    eval_set=[(X_val, y_bin_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

results['lgbm_binary'] = {
    'val'        : evaluate_binary(lgbm_binary, X_val,  y_bin_val,  'val',  'LGBM-Binary'),
    'test'       : evaluate_binary(lgbm_binary, X_test, y_bin_test, 'test', 'LGBM-Binary'),
    'best_params': best_lgb_bin,
}
models['lgbm_binary'] = lgbm_binary

y_prob_lgb = lgbm_binary.predict_proba(X_val)[:, 1]
fpr_lgb, tpr_lgb, _ = roc_curve(y_bin_val, y_prob_lgb)
roc_data['LightGBM'] = (fpr_lgb, tpr_lgb, results['lgbm_binary']['val']['roc_auc'])
print(f'\u2713 LightGBM Binary done in {time.time()-t0:.1f}s')

In [ ]:
# ── 4.4b  LightGBM Multiclass ─────────────────────────────────────────────────────────────
t0 = time.time()
print('\n► LightGBM Multiclass — HPO + Training')

X_lgb_mc, y_lgb_mc = stratified_subsample(X_mc_res, y_mc_res, n=SUBSAMPLE_N)

param_dist_lgb_mc = {
    'num_leaves'       : [31, 63, 127, 255],
    'learning_rate'    : [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    'n_estimators'     : [200, 300, 400, 500, 700],
    'subsample'        : [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree' : [0.6, 0.7, 0.8, 1.0],
    'min_child_samples': [10, 20, 50, 100],
    'reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'reg_lambda'       : [0, 0.1, 1.0, 5.0],
}
search_lgb_mc = RandomizedSearchCV(
    LGBMClassifier(class_weight='balanced', n_jobs=-1, random_state=SEED, verbose=-1),
    param_dist_lgb_mc, n_iter=10, cv=3,
    scoring='f1_macro', n_jobs=1, random_state=SEED, verbose=1
)
search_lgb_mc.fit(X_lgb_mc, y_lgb_mc)
best_lgb_mc = search_lgb_mc.best_params_
print(f'  Best params (LGBM multiclass): {best_lgb_mc}')

lgbm_multiclass = LGBMClassifier(
    **best_lgb_mc, class_weight='balanced', n_jobs=-1, random_state=SEED, verbose=-1
)
print(f'  Refitting on full resampled data ({len(y_mc_res):,} rows) with early stopping...')
lgbm_multiclass.fit(
    X_mc_res, y_mc_res,
    eval_set=[(X_val, y_mc_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

results['lgbm_multiclass'] = {
    'val'        : evaluate_multiclass(lgbm_multiclass, X_val,  y_mc_val,  'val',  'LGBM-Multiclass'),
    'test'       : evaluate_multiclass(lgbm_multiclass, X_test, y_mc_test, 'test', 'LGBM-Multiclass'),
    'best_params': best_lgb_mc,
}
models['lgbm_multiclass'] = lgbm_multiclass
print(f'\u2713 LightGBM Multiclass done in {time.time()-t0:.1f}s')

---
## § 4.5 — MLP (PyTorch)

**Architecture:**
```
37 → BatchNorm1d → [hidden layers] → ReLU → Dropout → num_classes
```

**Hyperparameter search:** 4 architecture configs × 5 epochs on 200 k subsample → pick best val F1 → full training (30 epochs, early stopping patience=5).

| | Binary | Multiclass |
|---|---|---|
| Loss | `BCEWithLogitsLoss(pos_weight=2.0)` | `CrossEntropyLoss(weight=inv_freq)` |
| Optimizer | Adam | Adam |
| Scheduler | `ReduceLROnPlateau(patience=3, factor=0.5)` | same |
| Batch size | 4096 | 4096 |

In [ ]:
# ── MLP class + training utilities ──────────────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.3):
        super().__init__()
        layers = [nn.BatchNorm1d(in_dim)]
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp(model, optimizer, scheduler, loader_train,
              X_val_t, y_val_t, criterion, task='binary',
              max_epochs=30, patience=5, save_path=None):
    """Train MLP with early stopping on val F1. Returns (model, history)."""
    model.to(DEVICE)
    best_val_f1, best_state, patience_ctr = -1.0, None, 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for Xb, yb in loader_train:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out.squeeze(), yb) if task == 'binary' else criterion(out, yb.long())
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        # Validation
        model.eval()
        with torch.no_grad():
            out_val = model(X_val_t.to(DEVICE))
        if task == 'binary':
            y_pred_val = (torch.sigmoid(out_val.squeeze()) > 0.5).cpu().numpy().astype(int)
            val_f1 = f1_score(y_val_t.numpy(), y_pred_val, zero_division=0)
        else:
            y_pred_val = out_val.argmax(dim=1).cpu().numpy()
            val_f1 = f1_score(y_val_t.numpy(), y_pred_val, average='macro', zero_division=0)

        avg_loss = epoch_loss / max(len(loader_train), 1)
        history.append({'epoch': epoch, 'loss': avg_loss, 'val_f1': val_f1})
        print(f'    Epoch {epoch:02d}/{max_epochs} | loss={avg_loss:.4f} | val_F1={val_f1:.4f}')

        scheduler.step(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'    Early stop at epoch {epoch}  (best val F1={best_val_f1:.4f})')
                break

    model.load_state_dict(best_state)
    if save_path:
        torch.save(best_state, save_path)
        print(f'    State dict saved → {save_path}')
    return model, history


class MLPPredictor:
    """Sklearn-compatible wrapper around a trained MLP nn.Module."""
    def __init__(self, model, task='binary'):
        self.model = model.eval()
        self.task  = task

    def predict(self, X):
        with torch.no_grad():
            out = self.model(torch.FloatTensor(X).to(DEVICE))
        if self.task == 'binary':
            return (torch.sigmoid(out.squeeze()) > 0.5).cpu().numpy().astype(int)
        return out.argmax(dim=1).cpu().numpy()

    def predict_proba(self, X):
        with torch.no_grad():
            out = self.model(torch.FloatTensor(X).to(DEVICE))
        if self.task == 'binary':
            p = torch.sigmoid(out.squeeze()).cpu().numpy()
            return np.stack([1 - p, p], axis=1)
        return torch.softmax(out, dim=1).cpu().numpy()


IN_DIM   = X_bin_res.shape[1]   # 37
BATCH_SZ = 4096

# Architecture candidates: (hidden_dims, lr, dropout)
MLP_CONFIGS = [
    ((256, 128),       1e-3, 0.3),
    ((512, 256, 128),  1e-3, 0.3),
    ((256, 128),       5e-4, 0.2),
    ((512, 256, 128),  5e-4, 0.3),
]
print(f'MLP in_dim={IN_DIM}, batch_size={BATCH_SZ}, {len(MLP_CONFIGS)} architecture configs.')

In [ ]:
# ── 4.5a  MLP Binary ───────────────────────────────────────────────────────────────────────
t0 = time.time()
print('\n► MLP Binary — Architecture Search (5 epochs each on 200k subsample)')

X_val_t_bin = torch.FloatTensor(X_val)
y_val_t_bin = torch.FloatTensor(y_bin_val)
pos_weight  = torch.tensor([2.0])

X_sub_bin, y_sub_bin = stratified_subsample(X_bin_res, y_bin_res, n=SUBSAMPLE_N)
sub_loader_bin = DataLoader(
    TensorDataset(torch.FloatTensor(X_sub_bin), torch.FloatTensor(y_sub_bin)),
    batch_size=BATCH_SZ, shuffle=True, num_workers=0
)

best_bin_score, best_bin_cfg = -1.0, None
for cfg in MLP_CONFIGS:
    h_dim, lr, dr = cfg
    m = MLP(IN_DIM, h_dim, 1, dropout=dr)
    opt = optim.Adam(m.parameters(), lr=lr)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
    m, hist = train_mlp(m, opt, sch, sub_loader_bin, X_val_t_bin, y_val_t_bin,
                        crit, task='binary', max_epochs=5, patience=5)
    score = hist[-1]['val_f1']
    print(f'  hidden={h_dim} lr={lr} dr={dr} → val_F1={score:.4f}')
    if score > best_bin_score:
        best_bin_score, best_bin_cfg = score, cfg

print(f'\nBest arch (binary): hidden={best_bin_cfg[0]} lr={best_bin_cfg[1]} dr={best_bin_cfg[2]}')
print('\n► MLP Binary — Full Training (30 epochs, early stopping patience=5)')

full_loader_bin = DataLoader(
    TensorDataset(torch.FloatTensor(X_bin_res), torch.FloatTensor(y_bin_res)),
    batch_size=BATCH_SZ, shuffle=True, num_workers=0
)
h_dim, lr_best, dr_best = best_bin_cfg
mlp_bin_model = MLP(IN_DIM, h_dim, 1, dropout=dr_best)
opt_bin  = optim.Adam(mlp_bin_model.parameters(), lr=lr_best)
sch_bin  = optim.lr_scheduler.ReduceLROnPlateau(opt_bin, patience=3, factor=0.5)
crit_bin = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))

mlp_bin_model, _ = train_mlp(
    mlp_bin_model, opt_bin, sch_bin, full_loader_bin,
    X_val_t_bin, y_val_t_bin, crit_bin,
    task='binary', max_epochs=30, patience=5,
    save_path=os.path.join(OUT_DIR, 'mlp_binary.pt')
)
mlp_binary = MLPPredictor(mlp_bin_model, task='binary')

results['mlp_binary'] = {
    'val'        : evaluate_binary(mlp_binary, X_val,  y_bin_val,  'val',  'MLP-Binary'),
    'test'       : evaluate_binary(mlp_binary, X_test, y_bin_test, 'test', 'MLP-Binary'),
    'best_arch'  : {'hidden_dims': list(h_dim), 'lr': lr_best, 'dropout': dr_best},
}
models['mlp_binary'] = mlp_binary

y_prob_mlp = mlp_binary.predict_proba(X_val)[:, 1]
fpr_mlp, tpr_mlp, _ = roc_curve(y_bin_val, y_prob_mlp)
roc_data['MLP'] = (fpr_mlp, tpr_mlp, results['mlp_binary']['val']['roc_auc'])
print(f'\u2713 MLP Binary done in {time.time()-t0:.1f}s')

In [ ]:
# ── 4.5b  MLP Multiclass ─────────────────────────────────────────────────────────────────
t0 = time.time()
NUM_CLASSES = 10
print('\n► MLP Multiclass — Architecture Search (5 epochs each on 200k subsample)')

# Class weights from resampled multiclass training data
counts_mc = np.bincount(y_mc_res.astype(int), minlength=NUM_CLASSES).astype(float)
class_wts  = 1.0 / (counts_mc + 1e-8)
class_wts  = class_wts / class_wts.sum() * NUM_CLASSES   # keep sum = num_classes
class_wts_t = torch.FloatTensor(class_wts)
print('Class weights:', {class_names[i]: round(float(class_wts[i]), 4) for i in range(NUM_CLASSES)})

X_val_t_mc = torch.FloatTensor(X_val)
y_val_t_mc = torch.LongTensor(y_mc_val.astype(int))

X_sub_mc, y_sub_mc = stratified_subsample(X_mc_res, y_mc_res, n=SUBSAMPLE_N)
sub_loader_mc = DataLoader(
    TensorDataset(torch.FloatTensor(X_sub_mc), torch.LongTensor(y_sub_mc.astype(int))),
    batch_size=BATCH_SZ, shuffle=True, num_workers=0
)

best_mc_score, best_mc_cfg = -1.0, None
for cfg in MLP_CONFIGS:
    h_dim, lr, dr = cfg
    m = MLP(IN_DIM, h_dim, NUM_CLASSES, dropout=dr)
    opt = optim.Adam(m.parameters(), lr=lr)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    crit = nn.CrossEntropyLoss(weight=class_wts_t.to(DEVICE))
    m, hist = train_mlp(m, opt, sch, sub_loader_mc, X_val_t_mc, y_val_t_mc,
                        crit, task='multiclass', max_epochs=5, patience=5)
    score = hist[-1]['val_f1']
    print(f'  hidden={h_dim} lr={lr} dr={dr} → val_F1_macro={score:.4f}')
    if score > best_mc_score:
        best_mc_score, best_mc_cfg = score, cfg

print(f'\nBest arch (multiclass): hidden={best_mc_cfg[0]} lr={best_mc_cfg[1]} dr={best_mc_cfg[2]}')
print('\n► MLP Multiclass — Full Training (30 epochs, early stopping patience=5)')

full_loader_mc = DataLoader(
    TensorDataset(torch.FloatTensor(X_mc_res), torch.LongTensor(y_mc_res.astype(int))),
    batch_size=BATCH_SZ, shuffle=True, num_workers=0
)
h_dim_mc, lr_mc, dr_mc = best_mc_cfg
mlp_mc_model = MLP(IN_DIM, h_dim_mc, NUM_CLASSES, dropout=dr_mc)
opt_mc  = optim.Adam(mlp_mc_model.parameters(), lr=lr_mc)
sch_mc  = optim.lr_scheduler.ReduceLROnPlateau(opt_mc, patience=3, factor=0.5)
crit_mc = nn.CrossEntropyLoss(weight=class_wts_t.to(DEVICE))

mlp_mc_model, _ = train_mlp(
    mlp_mc_model, opt_mc, sch_mc, full_loader_mc,
    X_val_t_mc, y_val_t_mc, crit_mc,
    task='multiclass', max_epochs=30, patience=5,
    save_path=os.path.join(OUT_DIR, 'mlp_multiclass.pt')
)
mlp_multiclass = MLPPredictor(mlp_mc_model, task='multiclass')

results['mlp_multiclass'] = {
    'val'        : evaluate_multiclass(mlp_multiclass, X_val,  y_mc_val,  'val',  'MLP-Multiclass'),
    'test'       : evaluate_multiclass(mlp_multiclass, X_test, y_mc_test, 'test', 'MLP-Multiclass'),
    'best_arch'  : {'hidden_dims': list(h_dim_mc), 'lr': lr_mc, 'dropout': dr_mc},
}
models['mlp_multiclass'] = mlp_multiclass
print(f'\u2713 MLP Multiclass done in {time.time()-t0:.1f}s')

---
## § 4.6 — Evaluation & Visualisations

All evaluation uses the **unaugmented** val and test sets.  

> ⚠️ Val attack ratio = **4.84%** vs Test attack ratio = **55.06%** — significant distribution shift. Results on both splits are reported.

In [ ]:
# ── Summary tables ─────────────────────────────────────────────────────────────────────

BINARY_KEYS = ['lr_binary', 'rf_binary', 'xgb_binary', 'lgbm_binary', 'mlp_binary']
MC_KEYS     = ['lr_multiclass', 'rf_multiclass', 'xgb_multiclass', 'lgbm_multiclass', 'mlp_multiclass']
LABELS      = ['LR', 'RF', 'XGBoost', 'LightGBM', 'MLP']

print('\n' + '='*90)
print('BINARY — VAL + TEST')
print('='*90)
print(f'{"Model":<14} {"Val Acc":>8} {"Val P":>7} {"Val R":>7} {"Val F1":>8} {"Val AUC":>9} {"Val FPR":>9} {"Test F1":>9}')
print('-'*90)
for key, lbl in zip(BINARY_KEYS, LABELS):
    v = results[key]['val']; t = results[key]['test']
    print(f'{lbl:<14} {v["accuracy"]:>8.4f} {v["precision"]:>7.4f} {v["recall"]:>7.4f} '
          f'{v["f1"]:>8.4f} {v["roc_auc"]:>9.4f} {v["fpr"]:>9.4f} {t["f1"]:>9.4f}')

print('\n' + '='*90)
print('MULTICLASS — VAL + TEST')
print('='*90)
print(f'{"Model":<14} {"Val Acc":>8} {"Val F1-mac":>11} {"Val F1-w":>10} {"Test F1-mac":>12} {"Test F1-w":>11}')
print('-'*90)
for key, lbl in zip(MC_KEYS, LABELS):
    v = results[key]['val']; t = results[key]['test']
    print(f'{lbl:<14} {v["accuracy"]:>8.4f} {v["f1_macro"]:>11.4f} '
          f'{v["f1_weighted"]:>10.4f} {t["f1_macro"]:>12.4f} {t["f1_weighted"]:>11.4f}')

best_bin = max(BINARY_KEYS, key=lambda k: results[k]['val']['f1'])
best_mc  = max(MC_KEYS,     key=lambda k: results[k]['val']['f1_macro'])
print(f'\n\u2605 Best binary   : {best_bin}  (val F1={results[best_bin]["val"]["f1"]:.4f})')
print(f'\u2605 Best multiclass: {best_mc}  (val F1-macro={results[best_mc]["val"]["f1_macro"]:.4f})')
print('\n\u26a0  Distribution shift: Val attack ratio=4.84% | Test attack ratio=55.06%')

In [ ]:
# ── ROC Curves (binary models, val set) ─────────────────────────────────────────────────
COLORS = ['steelblue', 'forestgreen', 'darkorange', 'crimson', 'mediumpurple']

fig, ax = plt.subplots(figsize=(8, 6))
for (name, (fpr, tpr, auc)), color in zip(roc_data.items(), COLORS):
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name}  (AUC={auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Binary Models (Val Set)')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'roc_curves.png'), dpi=150)
plt.show()
print('Saved roc_curves.png')

In [ ]:
# ── F1 Comparison Bar Chart ──────────────────────────────────────────────────────────────
bin_val  = [results[k]['val']['f1']       for k in BINARY_KEYS]
bin_test = [results[k]['test']['f1']      for k in BINARY_KEYS]
mc_val   = [results[k]['val']['f1_macro'] for k in MC_KEYS]
mc_test  = [results[k]['test']['f1_macro']for k in MC_KEYS]

x = np.arange(len(LABELS))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, val_f1, test_f1, title in [
    (axes[0], bin_val,  bin_test, 'Binary — F1 Score'),
    (axes[1], mc_val,   mc_test,  'Multiclass — F1 Macro'),
]:
    b1 = ax.bar(x - w/2, val_f1,  w * 0.9, label='Val',  color='steelblue',  alpha=0.85)
    b2 = ax.bar(x + w/2, test_f1, w * 0.9, label='Test', color='darkorange', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(LABELS)
    ax.set_ylim(0, 1.08); ax.set_ylabel('F1')
    ax.set_title(title); ax.legend(); ax.grid(axis='y', alpha=0.3)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Baseline Model F1 Comparison (Val vs Test)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'f1_comparison.png'), dpi=150)
plt.show()
print('Saved f1_comparison.png')

In [ ]:
# ── Confusion Matrices (best binary + best multiclass, val set) ──────────────────────────

# Best binary
y_pred_best_bin = models[best_bin].predict(X_val)
cm_bin = confusion_matrix(y_bin_val, y_pred_best_bin)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Best Binary CM ({best_bin}) — Val Set')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'cm_best_binary.png'), dpi=150)
plt.show()
print('Saved cm_best_binary.png')

# Best multiclass
y_pred_best_mc = models[best_mc].predict(X_val)
cm_mc = confusion_matrix(y_mc_val, y_pred_best_mc)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_mc, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax,
            annot_kws={'size': 7})
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Best Multiclass CM ({best_mc}) — Val Set')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'cm_best_multiclass.png'), dpi=150)
plt.show()
print('Saved cm_best_multiclass.png')

In [ ]:
# ── RF Feature Importance (top 20) ─────────────────────────────────────────────────────────
top20 = rf_importances.head(20)
colors_imp = ['crimson' if i < 5 else 'steelblue' for i in range(len(top20))]

fig, ax = plt.subplots(figsize=(8, 7))
top20.sort_values().plot(kind='barh', ax=ax,
                         color=colors_imp[::-1], edgecolor='white')
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('RF Feature Importance — Top 20 (Binary, Full Resampled Train)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'rf_importance.png'), dpi=150)
plt.show()
print('Saved rf_importance.png')

---
## § 4.7 — Serialisation

Outputs saved to `OUT_DIR` (`outputs/models/`):

| File | Content |
|---|---|
| `lr_binary.pkl` / `lr_multiclass.pkl` | Logistic Regression (joblib) |
| `rf_binary.pkl` / `rf_multiclass.pkl` | Random Forest (joblib) |
| `xgb_binary.pkl` / `xgb_multiclass.pkl` | XGBoost (joblib) |
| `lgbm_binary.pkl` / `lgbm_multiclass.pkl` | LightGBM (joblib) |
| `mlp_binary.pt` / `mlp_multiclass.pt` | MLP state dicts (saved during training) |
| `baseline_results.json` | All metrics — flat dict `{model: {val: {...}, test: {...}}}` |
| `phase4_results.md` | Markdown tables for appending to `outputs.md § Phase 4` |

In [ ]:
# ── Save sklearn / XGBoost / LightGBM models with joblib ─────────────────────────────────
sklearn_model_keys = [
    'lr_binary', 'lr_multiclass',
    'rf_binary', 'rf_multiclass',
    'xgb_binary', 'xgb_multiclass',
    'lgbm_binary', 'lgbm_multiclass',
]
for key in sklearn_model_keys:
    path = os.path.join(OUT_DIR, f'{key}.pkl')
    joblib.dump(models[key], path)
    print(f'Saved {key}.pkl')

# Confirm MLP state dicts
for tag in ('mlp_binary', 'mlp_multiclass'):
    p = os.path.join(OUT_DIR, f'{tag}.pt')
    status = '\u2714' if os.path.exists(p) else '\u2717 MISSING'
    print(f'{tag}.pt  {status}')

# ── Save baseline_results.json ─────────────────────────────────────────────────────────
json_path = os.path.join(OUT_DIR, 'baseline_results.json')
with open(json_path, 'w') as f:
    json.dump(_serialisable(results), f, indent=2)
print(f'\nSaved baseline_results.json → {json_path}')

In [ ]:
# ── Write phase4_results.md for appending to outputs.md ─────────────────────────────────

def fmt(v, key, fmt_str='.4f'):
    val = v.get(key, float('nan'))
    return f'{val:{fmt_str}}' if isinstance(val, float) else str(val)

lines = [
    '\n## \u00a7 Phase 4 \u2014 Baseline Models\n',
    '\n> Status: \u2705 Complete\n',
    '\n### 4.1 Binary Classification Results (Val Set)\n',
    '\n| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | FPR |\n',
    '|---|---|---|---|---|---|---|\n',
]
for key, lbl in zip(BINARY_KEYS, LABELS):
    v = results[key]['val']
    lines.append(f"| {lbl} | {fmt(v,'accuracy')} | {fmt(v,'precision')} | {fmt(v,'recall')} "
                 f"| {fmt(v,'f1')} | {fmt(v,'roc_auc')} | {fmt(v,'fpr')} |\n")

lines += [
    '\n### 4.2 Binary Classification Results (Test Set)\n',
    '\n| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | FPR |\n',
    '|---|---|---|---|---|---|---|\n',
]
for key, lbl in zip(BINARY_KEYS, LABELS):
    t = results[key]['test']
    lines.append(f"| {lbl} | {fmt(t,'accuracy')} | {fmt(t,'precision')} | {fmt(t,'recall')} "
                 f"| {fmt(t,'f1')} | {fmt(t,'roc_auc')} | {fmt(t,'fpr')} |\n")

lines += [
    '\n### 4.3 Multiclass Classification Results (Val Set)\n',
    '\n| Model | Accuracy | F1-Macro | F1-Weighted |\n',
    '|---|---|---|---|\n',
]
for key, lbl in zip(MC_KEYS, LABELS):
    v = results[key]['val']
    lines.append(f"| {lbl} | {fmt(v,'accuracy')} | {fmt(v,'f1_macro')} | {fmt(v,'f1_weighted')} |\n")

lines += [
    '\n### 4.4 Multiclass Classification Results (Test Set)\n',
    '\n| Model | Accuracy | F1-Macro | F1-Weighted |\n',
    '|---|---|---|---|\n',
]
for key, lbl in zip(MC_KEYS, LABELS):
    t = results[key]['test']
    lines.append(f"| {lbl} | {fmt(t,'accuracy')} | {fmt(t,'f1_macro')} | {fmt(t,'f1_weighted')} |\n")

# Per-class F1 for best multiclass model
lines += [
    f'\n### 4.5 Per-class F1 — Best Multiclass Model ({best_mc}, Val Set)\n',
    '\n| Class | F1 |\n', '|---|---|\n',
]
for cls, f1_val in results[best_mc]['val']['per_class_f1'].items():
    lines.append(f'| {cls} | {f1_val:.4f} |\n')

lines += [
    '\n### 4.6 Best Baseline Models\n',
    f'\n- **Binary:**     `{best_bin}`  (val F1={results[best_bin]["val"]["f1"]:.4f}, test F1={results[best_bin]["test"]["f1"]:.4f})\n',
    f'- **Multiclass:** `{best_mc}`  (val F1-macro={results[best_mc]["val"]["f1_macro"]:.4f}, test F1-macro={results[best_mc]["test"]["f1_macro"]:.4f})\n',
    '\n> ⚠\ufe0f  Distribution shift: Val attack ratio=4.84% | Test attack ratio=55.06%\n',
    '\n### 4.7 Plots Generated\n',
    '\n- [x] `outputs/models/roc_curves.png`\n',
    '- [x] `outputs/models/f1_comparison.png`\n',
    '- [x] `outputs/models/cm_best_binary.png`\n',
    '- [x] `outputs/models/cm_best_multiclass.png`\n',
    '- [x] `outputs/models/rf_importance.png`\n',
    '- [x] `outputs/models/baseline_results.json`\n',
]

md_path = os.path.join(OUT_DIR, 'phase4_results.md')
with open(md_path, 'w', encoding='utf-8') as f:
    f.writelines(lines)
print(f'Saved phase4_results.md → {md_path}')
print('Append the contents of this file to outputs.md § Phase 4.')

# If running locally, also update outputs.md in-place
if PLATFORM == 'local':
    local_outputs_md = os.path.join(BASE, 'outputs.md')
    if os.path.exists(local_outputs_md):
        with open(local_outputs_md, 'r', encoding='utf-8') as f:
            content = f.read()
        marker = '## \u00a7 Phase 4 \u2014 Baseline Models'
        placeholder = '> Status: \U0001f532 Not Started'
        new_section = ''.join(lines)
        if placeholder in content:
            # Replace the skeleton section
            start = content.find(marker)
            if start != -1:
                # Find next top-level section or end-of-file
                next_sec = content.find('\n## \u00a7 Phase', start + 10)
                end = next_sec if next_sec != -1 else len(content)
                content = content[:start] + new_section.strip() + '\n\n' + content[end:]
                with open(local_outputs_md, 'w', encoding='utf-8') as f:
                    f.write(content)
                print(f'Updated outputs.md → {local_outputs_md}')
        else:
            print('outputs.md Phase 4 section already updated or marker not found.')

In [ ]:
# ── Final output directory listing ────────────────────────────────────────────────────────
print('\n=== Outputs saved to OUT_DIR ===')
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fname:<40} {size_kb:>8.1f} KB')

print('\n=== Phase 4 Complete ===')
print(f'  Binary winner   : {best_bin}  '
      f'(val F1={results[best_bin]["val"]["f1"]:.4f}  '
      f'test F1={results[best_bin]["test"]["f1"]:.4f})')
print(f'  Multiclass winner: {best_mc}  '
      f'(val F1-mac={results[best_mc]["val"]["f1_macro"]:.4f}  '
      f'test F1-mac={results[best_mc]["test"]["f1_macro"]:.4f})')
print('\nNext: Phase 5 — GNN Training (05_gnn.ipynb)')